# Analisi delle risposte ANITA

Visualizza le predizioni dell'LLM (token complessi + motivazioni) per le frasi del test set.

In [1]:
import pickle

with open("/code/HLTproject_code/task_1B_complexity_explanation/data/eval_set_full.pkl", "rb") as f:
    eval_set = pickle.load(f)
with open("/code/HLTproject_code/task_1B_complexity_explanation/anita_results/parsed_anita_full.pkl", "rb") as f:
    resp = pickle.load(f)

for i, (ex, r) in enumerate(zip(eval_set, resp)):
    print(f"───── esempio {i} ─────")
    print("FRASE:", " ".join(ex["tokens"]))
    print("RISPOSTA:", r)
    print()

───── esempio 0 ─────
FRASE: Prodotto dalla BBC , il film esce solo nel 1998 ed ottiene numerosi riconoscimenti internazionali , tra cui la candidatura al Premio Oscar per il miglior cortometraggio animato .
RISPOSTA: {'complex': [17, 23, 24, 25], 'motivi': {'17': 'inciso che separa, si può eliminare la ripetizione', '23': "preposizione e articolo, si può dire 'per il miglior'"}}

───── esempio 1 ─────
FRASE: ad aprile per la fiera del bestiame , a giugno per la compravendita del grano , a settembre per i festeggiamenti del santo patrono , con i fuochi pirotecnici e con delle corse ippiche che richiamavano dai paesi circostanti una moltitudine di gente .
RISPOSTA: {'complex': [35, 36, 37, 38], 'motivi': {'36': 'preposizione con funzione di ripetizione, si può eliminare', '35': "verbo complesso, si può dire 'attraevano'", '37': "riepilica di 'circostanti', si può dire 'vicini'"}}

───── esempio 2 ─────
FRASE: La popolarità e il prestigio di Marco Antonio peraltro non soffrirono a causa 

In [2]:
import pickle, sys
import numpy as np
from IPython.display import HTML, display

sys.path.insert(0, '/code/HLTproject_code/task_1B_complexity_explanation')
from utils.parse_responses import extract_annotation

BASE = '/code/HLTproject_code/task_1B_complexity_explanation'

with open(f'{BASE}/data/eval_set_full.pkl', 'rb') as f:
    eval_set = pickle.load(f)
with open(f'{BASE}/anita_results/anita_resp_full.pkl', 'rb') as f:
    anita_raw = pickle.load(f)

# parsifica tutte le risposte una volta sola
parsed_all = []
stats = {}
for ex, raw in zip(eval_set, anita_raw):
    p, status = extract_annotation(raw, n_tokens=len(ex['tokens']))
    parsed_all.append(p)
    stats[status] = stats.get(status, 0) + 1

print(f'Frasi caricate: {len(eval_set)}')
print('Parsing status:', stats)

Frasi caricate: 7539
Parsing status: {'recovered_regex': 2760, 'ok': 4779}


In [3]:
# ── funzione di rendering ──────────────────────────────────────────────────────

COL_ANITA  = '#DDA0DD'   # viola: token che ANITA ritiene complessi
COL_SILVER = '#90EE90'   # verde: silver label (ground truth)
COL_BOTH   = '#FFB347'   # arancione: complesso sia per ANITA sia per silver


def render_example(i, show_silver=True):
    """
    Mostra la frase i con:
    - token evidenziati in base a ANITA (viola) e silver (verde)
    - tooltip al passaggio del mouse con il motivo per i token ANITA
    - elenco motivi sotto la frase
    - statistiche sintetiche
    """
    ex     = eval_set[i]
    parsed = parsed_all[i]
    toks   = ex['tokens']
    silver = ex['silver']         # lista binaria 0/1
    simp   = ex['simp_tokens']

    complex_set = set(parsed['complex']) if parsed else set()
    motivi      = parsed.get('motivi', {}) if parsed else {}

    # ── costruzione HTML della frase ────────────────────────────────────────
    token_spans = []
    for j, tok in enumerate(toks):
        in_anita  = j in complex_set
        in_silver = bool(silver[j]) if show_silver else False
        motivo    = motivi.get(str(j), '')

        if in_anita and in_silver:
            bg = COL_BOTH
        elif in_anita:
            bg = COL_ANITA
        elif in_silver:
            bg = COL_SILVER
        else:
            bg = None

        style = (
            f'background:{bg};padding:1px 4px;border-radius:3px;'
            'margin:1px;display:inline-block;cursor:default;'
        ) if bg else 'margin:1px;display:inline-block;'

        # numero indice sopra il token (solo se complesso per ANITA)
        sup = f'<sup style="font-size:0.65em;color:#888">{j}</sup>' if in_anita else ''

        # tooltip con il motivo
        title = f' title="[{j}] {motivo}"' if motivo else (f' title="[{j}]"' if in_anita else '')

        token_spans.append(f'<span style="{style}"{title}>{tok}{sup}</span>')

    sentence_html = ' '.join(token_spans)

    # ── elenco motivi ────────────────────────────────────────────────────────
    if motivi:
        motivi_html = '<ul style="margin:4px 0 0 16px;font-size:0.9em;color:#444">'
        for idx_s, mot in sorted(motivi.items(), key=lambda x: int(x[0])):
            idx_i = int(idx_s)
            tok_label = toks[idx_i] if idx_i < len(toks) else f'?{idx_s}'
            motivi_html += (
                f'<li><b style="background:{COL_ANITA};padding:1px 3px;border-radius:2px">'
                f'[{idx_s}] {tok_label}</b>: {mot}</li>'
            )
        motivi_html += '</ul>'
    else:
        motivi_html = '<p style="color:#aaa;font-size:0.85em;margin:4px 0">'\
                      '(nessun motivo fornito)</p>'

    # ── statistiche ─────────────────────────────────────────────────────────
    n_anita  = len(complex_set)
    n_silver = sum(silver)
    n_both   = len(complex_set & {j for j, s in enumerate(silver) if s})
    stat_str = (
        f'ANITA: <b>{n_anita}</b> token &nbsp;|&nbsp; '
        f'Silver: <b>{n_silver}</b> token &nbsp;|&nbsp; '
        f'Overlap: <b>{n_both}</b>'
    )

    # ── legenda ──────────────────────────────────────────────────────────────
    legend = (
        f'<span style="background:{COL_ANITA};padding:1px 5px;border-radius:3px">ANITA</span>&nbsp;'
        f'<span style="background:{COL_SILVER};padding:1px 5px;border-radius:3px">silver</span>&nbsp;'
        f'<span style="background:{COL_BOTH};padding:1px 5px;border-radius:3px">entrambi</span>'
    )

    html = f"""
    <div style="border:1px solid #ddd;padding:12px 16px;margin:10px 0;
                border-radius:8px;font-family:sans-serif">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px">
        <span style="font-weight:bold;font-size:1.05em">Esempio {i}
          <span style="font-weight:normal;color:#888;font-size:0.85em">(id={ex['idx']})</span>
        </span>
        <span style="font-size:0.85em;color:#555">{legend}</span>
      </div>

      <div style="margin:6px 0;line-height:2.0">{sentence_html}</div>

      <div style="margin-top:4px;font-size:0.82em;color:#666">
        <b>Semplificata:</b> <i>{
          ' '.join(simp)
        }</i>
      </div>

      <div style="margin-top:8px">
        <b style="font-size:0.9em">Motivazioni ANITA:</b>
        {motivi_html}
      </div>

      <div style="margin-top:8px;font-size:0.82em;color:#555">
        {stat_str}
      </div>
    </div>
    """
    display(HTML(html))


print('render_example() pronta.')

render_example() pronta.


## Scegli un esempio specifico

Cambia `INDICE` con il numero che vuoi (0 – 7538).

In [4]:
INDICE = 3
render_example(INDICE)

## Mostra N esempi consecutivi

In [5]:
INIZIO = 0
N      = 10

for i in range(INIZIO, INIZIO + N):
    render_example(i)

## Mostra una lista di indici a scelta

In [6]:
INDICI = [0, 3, 7, 15]

for i in INDICI:
    render_example(i)

## Campione casuale

In [7]:
N_CAMPIONE = 5
rng = np.random.default_rng()   # seed casuale; metti seed=42 per riproducibilità

for i in rng.choice(len(eval_set), size=N_CAMPIONE, replace=False).tolist():
    render_example(i)